# Lesson 9: AI Incident Management
## Exercise: Threshold-Based Incident Detection for TransitAI

**Quick Reference — Incident Management Concepts:**

| Concept | Description |
|---------|-------------|
| **Severity Levels** | S1 (Critical, <15min) → S2 (High, <1hr) → S3 (Medium, <4hr) → S4 (Low, <24hr) |
| **Detection Methods** | Threshold-based, anomaly detection, user reports, automated monitoring |
| **Response Phases** | Detect → Triage → Classify → Contain → Investigate → Resolve → Review |
| **Key Metrics** | Model accuracy, response latency, bias score, error rate, uptime |
| **Escalation Triggers** | Safety risk, data breach, widespread impact, regulatory implication |

**Scenario:** TransitAI operates autonomous route optimization across 15 metro areas serving 2M+ daily passengers. You'll implement incident detection logic for their monitoring system.

**Your Task:** Complete 3 TODO sections that require incident management thinking (~10-15 min).

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("TransitAI Incident Detection System — Libraries loaded")

## Monitoring Data (Pre-loaded)
100 hours of TransitAI system monitoring data with 7 injected anomalies simulating real incidents.

In [ ]:
# Generate 100 hours of monitoring data
n_samples = 100
timestamps = [datetime.now() - timedelta(hours=100-i) for i in range(n_samples)]

monitoring_data = pd.DataFrame({
    'timestamp': timestamps,
    'model_accuracy': np.random.normal(94, 2, n_samples).clip(70, 100),
    'response_latency_ms': np.random.normal(120, 15, n_samples).clip(50, 500),
    'bias_score': np.random.normal(0.05, 0.02, n_samples).clip(0, 1),
    'error_rate': np.random.normal(0.02, 0.005, n_samples).clip(0, 1),
    'uptime_pct': np.random.normal(99.9, 0.1, n_samples).clip(95, 100),
    'content_safety_score': np.random.normal(0.98, 0.01, n_samples).clip(0, 1)
})

# Inject realistic anomalies
anomalies = [
    {'name': 'Accuracy Drop (Performance)', 'start': 10, 'end': 15, 'metric': 'model_accuracy', 'value': 78, 'severity': 'S2'},
    {'name': 'Latency Spike (Load)', 'start': 22, 'end': 27, 'metric': 'response_latency_ms', 'value': 280, 'severity': 'S2'},
    {'name': 'Bias Score Increase', 'start': 35, 'end': 40, 'metric': 'bias_score', 'value': 0.25, 'severity': 'S2'},
    {'name': 'Error Rate Spike', 'start': 50, 'end': 55, 'metric': 'error_rate', 'value': 0.15, 'severity': 'S3'},
    {'name': 'Uptime Drop (Outage)', 'start': 65, 'end': 68, 'metric': 'uptime_pct', 'value': 96.5, 'severity': 'S1'},
    {'name': 'Accuracy + Latency Combined', 'start': 80, 'end': 85, 'metric': 'model_accuracy', 'value': 82, 'severity': 'S2'},
    {'name': 'Latency Spike #2', 'start': 80, 'end': 85, 'metric': 'response_latency_ms', 'value': 250, 'severity': 'S2'},
    {'name': 'Content Safety Drop (Hallucination)', 'start': 42, 'end': 47, 'metric': 'content_safety_score', 'value': 0.88, 'severity': 'S2'},
]

for anomaly in anomalies:
    monitoring_data.loc[anomaly['start']:anomaly['end'], anomaly['metric']] = anomaly['value'] + np.random.normal(0, 1, anomaly['end'] - anomaly['start'] + 1)

print(f"Monitoring data: {len(monitoring_data)} hours, {len(monitoring_data.columns)-1} metrics")
print(f"Injected anomalies: {len(anomalies)}")
monitoring_data.head(3)

## Detection Thresholds (Pre-defined)

In [ ]:
thresholds = {
    'model_accuracy': {'warning': 90, 'critical': 85, 'direction': 'below', 'unit': '%'},
    'response_latency_ms': {'warning': 180, 'critical': 250, 'direction': 'above', 'unit': 'ms'},
    'bias_score': {'warning': 0.10, 'critical': 0.20, 'direction': 'above', 'unit': 'score'},
    'error_rate': {'warning': 0.05, 'critical': 0.10, 'direction': 'above', 'unit': 'rate'},
    'uptime_pct': {'warning': 99.5, 'critical': 99.0, 'direction': 'below', 'unit': '%'},
    'content_safety_score': {'warning': 0.95, 'critical': 0.90, 'direction': 'below', 'unit': 'score'},
}

print("Detection Thresholds:")
for metric, t in thresholds.items():
    print(f"  {metric}: warning={t['warning']}{t['unit']}, critical={t['critical']}{t['unit']} ({t['direction']})")

## TODO #1: Implement Incident Detection Logic (All 4 Severity Tiers)

Complete the `detect_incidents()` function that detects incidents across ALL 4 severity levels (S1-S4):

**For each data point:**
1. Compare the metric value against warning and critical thresholds
2. Check the `direction` field — 'below' means alert when value < threshold, 'above' means alert when value > threshold
3. **S1 (Critical):** Value exceeds critical threshold — immediate safety/system risk
4. **S2 (High):** Compound metric breaches where 2+ metrics simultaneously exceed warning thresholds OR 1+ critical + 1+ warning = systemic degradation, not isolated
5. **S3 (Medium):** Single warning-level breach — degraded but not critical
6. **S4 (Low):** Metrics approaching thresholds — within 10% of warning threshold for 3+ consecutive hours

**Hints for S2 detection:**
- S2 requires multiple metrics breaching at the same timestamp (e.g., model_accuracy AND response_latency both in warning range)
- This indicates systemic issues rather than isolated problems
- Example: If accuracy drops AND latency spikes together, it's likely a load issue affecting the whole system

**Hints for S4 detection:**
- S4 is advisory-level: metric is approaching warning but hasn't breached yet
- Track metrics within 10% of the warning threshold
- Useful for proactive alerting before issues escalate

**Governance thinking:** Why do we use four severity levels instead of just two? How does this support different response strategies?

In [ ]:
def detect_incidents(data, thresholds):
    """
    Detect incidents by comparing metrics against thresholds.
    
    Args:
        data: DataFrame with monitoring metrics
        thresholds: Dict with warning/critical thresholds per metric
    
    Returns:
        List of incident dicts with: timestamp, metric, value, threshold_type, threshold_value, severity
    """
    incidents = []
    
    for idx, row in data.iterrows():
        for metric, thresh in thresholds.items():
            value = row[metric]
            
            # TODO: Implement threshold comparison logic
            # Check direction ('below' or 'above') to determine comparison
            # Compare against critical threshold first (higher severity), then warning
            # Append incident dict with: timestamp, metric, value, threshold_type, threshold_value, severity
            pass
    
    return incidents

# Run detection
detected = detect_incidents(monitoring_data, thresholds)
print(f"Detected {len(detected)} incidents")

## Incident Report Generation (Pre-provided)

In [ ]:
if detected:
    incident_df = pd.DataFrame(detected)
    incident_df = incident_df.sort_values('timestamp')
    
    print("=" * 70)
    print("TRANSITAI INCIDENT REPORT")
    print("=" * 70)
    print(f"\nTotal incidents detected: {len(incident_df)}")
    print(f"\nBy severity:")
    # Note: Your detection logic should find S1, S2, S3, and possibly S4 incidents
    for sev in ['S1', 'S2', 'S3', 'S4']:
        count = len(incident_df[incident_df['severity'] == sev])
        if count > 0:
            sev_map = {'S1': 'Critical', 'S2': 'High', 'S3': 'Medium', 'S4': 'Low'}
            print(f"  {sev} ({sev_map.get(sev, 'Unknown')}): {count}")
    print(f"\nBy metric:")
    for metric in incident_df['metric'].unique():
        count = len(incident_df[incident_df['metric'] == metric])
        print(f"  {metric}: {count}")
    
    incident_df.head(10)
else:
    print("No incidents detected — check your detection logic!")
    incident_df = pd.DataFrame()

## TODO #2: Implement Escalation Decision Logic (S1-S4)

Complete the `determine_escalation()` function that decides WHO should be notified based on:
- The incident severity (S1 = critical, S2 = high, S3 = medium, S4 = low)
- The metric type (some metrics like uptime and content_safety are more operationally/ethically critical)

**Severity escalation guidelines:**
- **S1 (Critical):** C-suite, legal, regulatory bodies, emergency response. Activate war room. <15 min SLA
- **S2 (High):** VPs and managers, potentially regulatory. <1 hour SLA
- **S3 (Medium):** Engineering leads and team managers. <4 hours SLA
- **S4 (Low):** Development team. <24 hours SLA

**Metric-specific considerations:**
- `bias_score`: Always notify Ethics Review Board if S2+. Regulatory implications for transit systems
- `content_safety_score`: Always notify Ethics Review Board AND Content Safety Team. May require regulatory notification
- `uptime_pct`: Always notify Operations Center. Emergency Response Team for S1

**Governance thinking:** In a safety-critical transit system, who needs to know about different types of incidents? Consider regulatory reporting requirements (NTSB/FTA for transit).

In [ ]:
def determine_escalation(severity, metric):
    """
    Determine escalation path based on severity and metric type.
    
    Args:
        severity: 'S1', 'S2', 'S3', or 'S4'
        metric: Name of the affected metric
    
    Returns:
        Dict with: notify_list, response_sla, requires_war_room (bool), regulatory_report (bool)
    """
    # TODO: Implement escalation logic
    # S1 (Critical): Notify C-suite, activate war room, regulatory reporting
    # S2 (High): Notify VPs and managers, potential regulatory
    # S3 (Medium): Notify team leads
    # S4 (Low): Team-level only
    # Consider: bias_score and uptime_pct incidents may need different escalation than latency
    
    return {
        'notify_list': [],
        'response_sla': '',
        'requires_war_room': False,
        'regulatory_report': False
    }

# Apply to detected incidents
if len(incident_df) > 0:
    escalations = incident_df.apply(
        lambda row: determine_escalation(row['severity'], row['metric']), axis=1
    )
    escalation_df = pd.DataFrame(escalations.tolist())
    result_df = pd.concat([incident_df.reset_index(drop=True), escalation_df], axis=1)
    print(f"Escalation decisions for {len(result_df)} incidents:")
    result_df[['metric', 'severity', 'notify_list', 'response_sla', 'requires_war_room', 'regulatory_report']].head(10)

## TODO #3: Generate Response Recommendations

Complete the `recommend_response()` function that suggests specific response actions based on the incident type (metric) and severity (S1-S4).

**Governance thinking:** What's the difference between containing an incident and resolving it? Why is the distinction important for a safety-critical transit system?

**For each metric, define:**
- **containment_actions:** What do you do IMMEDIATELY to stop the bleeding? (e.g., disable the feature, failover, escalate)
- **resolution_steps:** What do you do to FIX the root cause? (e.g., patch, retrain, scale)
- **post_incident:** What do you do to PREVENT recurrence? (e.g., add monitoring, testing, guardrails)

**Key insights on compound incidents:**
- When `model_accuracy` drops AND `error_rate` spikes simultaneously, the system is experiencing compound failure
- Compound incidents may require more aggressive containment: "activate full system failover" rather than individual metric responses
- The root cause is likely systemic (load, corruption, cascading failure) not isolated to one metric
- Consider: What other metric pairs might indicate systemic issues?
  - accuracy + latency together = overload
  - accuracy + error_rate together = data corruption or model instability
  - content_safety_score + error_rate = content generation system malfunction

**Severity adjustments:**
- S1: Add war room activation and regulatory notification
- S2: Alert management team; compound incidents may need failover consideration
- S4: Simplified response; focus on monitoring rather than immediate action

In [ ]:
def recommend_response(metric, severity):
    """
    Recommend response actions based on incident type and severity.
    
    Args:
        metric: The affected metric name
        severity: 'S1', 'S2', 'S3', or 'S4'
    
    Returns:
        Dict with: containment_actions (list), resolution_steps (list), post_incident (list)
    """
    # TODO: Define response recommendations per metric type
    # For each metric, think about:
    # - containment_actions: What do you do IMMEDIATELY to stop the bleeding?
    # - resolution_steps: What do you do to FIX the root cause?
    # - post_incident: What do you do to PREVENT recurrence?
    # Consider how severity affects urgency and scope of response
    
    return {
        'containment_actions': [],
        'resolution_steps': [],
        'post_incident': []
    }

# Generate recommendations for unique incident types
if len(incident_df) > 0:
    unique_incidents = incident_df.drop_duplicates(subset=['metric', 'severity'])
    print("Response Recommendations:")
    print("=" * 70)
    for _, row in unique_incidents.iterrows():
        rec = recommend_response(row['metric'], row['severity'])
        print(f"\n{row['metric']} ({row['severity']}):")
        print(f"  Contain: {rec['containment_actions']}")
        print(f"  Resolve: {rec['resolution_steps']}")
        print(f"  Prevent: {rec['post_incident']}")

## Visualization (Pre-provided)

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(14, 14))
fig.suptitle('TransitAI System Monitoring — Incident Detection', fontsize=14, fontweight='bold')

metrics = ['model_accuracy', 'response_latency_ms', 'bias_score', 'error_rate', 'uptime_pct', 'content_safety_score']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#e377c2']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax = axes[i]
    ax.plot(range(len(monitoring_data)), monitoring_data[metric], color=color, linewidth=1, alpha=0.8)
    
    t = thresholds[metric]
    ax.axhline(y=t['warning'], color='orange', linestyle='--', alpha=0.7, label=f"Warning ({t['warning']})")
    ax.axhline(y=t['critical'], color='red', linestyle='--', alpha=0.7, label=f"Critical ({t['critical']})")
    
    # Mark detected incidents
    if len(incident_df) > 0:
        metric_incidents = incident_df[incident_df['metric'] == metric]
        for _, inc in metric_incidents.iterrows():
            idx = monitoring_data[monitoring_data['timestamp'] == inc['timestamp']].index
            if len(idx) > 0:
                marker = 'v' if inc['severity'] == 'S1' else '^' if inc['severity'] == 'S2' else 's' if inc['severity'] == 'S3' else 'D'
                inc_color = 'red' if inc['severity'] == 'S1' else 'orange' if inc['severity'] == 'S2' else 'yellow' if inc['severity'] == 'S3' else 'lightblue'
                ax.plot(idx[0], inc['value'], marker, color=inc_color, markersize=8)
    
    ax.set_ylabel(metric.replace('_', ' ').title(), fontsize=9)
    ax.legend(loc='upper right', fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('transitai_monitoring.png', dpi=100, bbox_inches='tight')
plt.show()
print("Visualization saved as transitai_monitoring.png")

## Summary

You implemented three key components of an incident detection system:
1. **Detection logic** — threshold-based monitoring with warning/critical levels
2. **Escalation decisions** — routing incidents to the right people based on severity
3. **Response recommendations** — containment, resolution, and prevention actions

In a production transit system, these would integrate with real-time monitoring tools (Prometheus, DataDog) and incident management platforms (PagerDuty, Jira).